# Go Concurrency Patterns

**Objective:** Master goroutines, channels, select, WaitGroups, and common concurrency patterns — the feature that makes Go ideal for high-throughput services.

**Prerequisites:** Go basics (notebook 01)

## 1. Goroutines

In [ ]:
import (
    "fmt"
    "time"
)

// Basic goroutine
go func() {
    fmt.Println("Hello from goroutine!")
}()

// Multiple goroutines
for i := 0; i < 5; i++ {
    go func(n int) {
        fmt.Printf("Goroutine %d running\n", n)
    }(i)
}

time.Sleep(100 * time.Millisecond)
fmt.Println("All goroutines launched")

## 2. Channels (Unbuffered and Buffered)

In [ ]:
import "fmt"

// Unbuffered channel — sender blocks until receiver is ready
ch := make(chan string)

go func() {
    ch <- "hello from goroutine"
}()

msg := <-ch
fmt.Println("Received:", msg)

// Buffered channel — can hold values without a receiver
buf := make(chan int, 3)
buf <- 1
buf <- 2
buf <- 3
fmt.Printf("Buffered channel: len=%d, cap=%d\n", len(buf), cap(buf))
fmt.Println("Values:", <-buf, <-buf, <-buf)

// Channel direction in function signatures
// func produce(out chan<- int) — send only
// func consume(in <-chan int)  — receive only

## 3. Select Statement

In [ ]:
import (
    "fmt"
    "time"
)

ch1 := make(chan string)
ch2 := make(chan string)

go func() {
    time.Sleep(50 * time.Millisecond)
    ch1 <- "result from service A"
}()

go func() {
    time.Sleep(30 * time.Millisecond)
    ch2 <- "result from service B"
}()

// Select waits on multiple channels — whichever is ready first wins
for i := 0; i < 2; i++ {
    select {
    case msg := <-ch1:
        fmt.Println("ch1:", msg)
    case msg := <-ch2:
        fmt.Println("ch2:", msg)
    case <-time.After(100 * time.Millisecond):
        fmt.Println("timeout!")
    }
}

## 4. WaitGroups and Mutexes

In [ ]:
import (
    "fmt"
    "sync"
)

// WaitGroup — wait for a group of goroutines to finish
var wg sync.WaitGroup

for i := 0; i < 5; i++ {
    wg.Add(1)
    go func(n int) {
        defer wg.Done()
        fmt.Printf("Worker %d done\n", n)
    }(i)
}

wg.Wait()
fmt.Println("All workers finished")

// Mutex — protect shared state
var mu sync.Mutex
counter := 0
var wg2 sync.WaitGroup

for i := 0; i < 1000; i++ {
    wg2.Add(1)
    go func() {
        defer wg2.Done()
        mu.Lock()
        counter++
        mu.Unlock()
    }()
}

wg2.Wait()
fmt.Printf("Counter (with mutex): %d (expected 1000)\n", counter)

## 5. Worker Pool Pattern

In [ ]:
import (
    "fmt"
    "sync"
)

jobs := make(chan int, 10)
results := make(chan int, 10)

// Start 3 workers
var wg sync.WaitGroup
for w := 0; w < 3; w++ {
    wg.Add(1)
    go func(id int) {
        defer wg.Done()
        for job := range jobs {
            result := job * job
            fmt.Printf("Worker %d: %d² = %d\n", id, job, result)
            results <- result
        }
    }(w)
}

// Send jobs
for j := 1; j <= 9; j++ {
    jobs <- j
}
close(jobs)

// Wait for workers then close results
go func() {
    wg.Wait()
    close(results)
}()

// Collect results
total := 0
for r := range results {
    total += r
}
fmt.Printf("Sum of squares: %d\n", total)

## 6. Fan-In / Fan-Out Pattern

In [ ]:
import (
    "fmt"
    "sync"
)

// Fan-out: one source, multiple processors
func generate(nums ...int) <-chan int {
    out := make(chan int)
    go func() {
        for _, n := range nums {
            out <- n
        }
        close(out)
    }()
    return out
}

func square(in <-chan int) <-chan int {
    out := make(chan int)
    go func() {
        for n := range in {
            out <- n * n
        }
        close(out)
    }()
    return out
}

// Fan-in: merge multiple channels into one
func merge(channels ...<-chan int) <-chan int {
    out := make(chan int)
    var wg sync.WaitGroup
    for _, ch := range channels {
        wg.Add(1)
        go func(c <-chan int) {
            defer wg.Done()
            for v := range c {
                out <- v
            }
        }(ch)
    }
    go func() {
        wg.Wait()
        close(out)
    }()
    return out
}

// Pipeline: generate → fan-out to 2 squarers → fan-in
source := generate(1, 2, 3, 4, 5, 6, 7, 8)

// Fan-out to two workers (they'll split the work)
ch1 := square(source)
// Note: in real code you'd split the source; here we demo merge
merged := merge(ch1)

for v := range merged {
    fmt.Print(v, " ")
}
fmt.Println()

## 7. Context (Cancellation, Timeout, Deadline)

In [ ]:
import (
    "context"
    "fmt"
    "time"
)

// Cancel a goroutine from outside
ctx, cancel := context.WithCancel(context.Background())

go func(ctx context.Context) {
    for {
        select {
        case <-ctx.Done():
            fmt.Println("Worker cancelled:", ctx.Err())
            return
        default:
            fmt.Println("Working...")
            time.Sleep(20 * time.Millisecond)
        }
    }
}(ctx)

time.Sleep(60 * time.Millisecond)
cancel()
time.Sleep(10 * time.Millisecond)

// Timeout context — auto-cancels after duration
ctx2, cancel2 := context.WithTimeout(context.Background(), 50*time.Millisecond)
defer cancel2()

select {
case <-time.After(100 * time.Millisecond):
    fmt.Println("Operation completed")
case <-ctx2.Done():
    fmt.Println("Timed out:", ctx2.Err())
}

## 8. Rate Limiting

In [ ]:
import (
    "fmt"
    "time"
)

// Basic rate limiter: 1 event per 100ms using a ticker
limiter := time.NewTicker(100 * time.Millisecond)
defer limiter.Stop()

requests := []int{1, 2, 3, 4, 5}

start := time.Now()
for _, req := range requests {
    <-limiter.C // blocks until next tick
    fmt.Printf("Request %d at %v\n", req, time.Since(start).Round(time.Millisecond))
}

// Bursty rate limiter: allow short bursts then rate-limit
fmt.Println("\n--- Bursty limiter ---")
burstyLimiter := make(chan time.Time, 3)

// Pre-fill buffer for initial burst
for i := 0; i < 3; i++ {
    burstyLimiter <- time.Now()
}

// Refill at steady rate
go func() {
    for t := range time.Tick(100 * time.Millisecond) {
        burstyLimiter <- t
    }
}()

burstyRequests := []int{1, 2, 3, 4, 5, 6}
start2 := time.Now()
for _, req := range burstyRequests {
    <-burstyLimiter
    fmt.Printf("Bursty request %d at %v\n", req, time.Since(start2).Round(time.Millisecond))
}

## 9. Semaphore (Bounded Concurrency)

In [ ]:
import (
    "fmt"
    "sync"
    "time"
)

// Use a buffered channel as a counting semaphore to limit
// concurrent goroutines (e.g., max 3 at a time for 10 tasks)
maxConcurrency := 3
sem := make(chan struct{}, maxConcurrency)

var wg sync.WaitGroup
start := time.Now()

for i := 1; i <= 10; i++ {
    wg.Add(1)
    go func(id int) {
        defer wg.Done()
        sem <- struct{}{}        // acquire slot
        defer func() { <-sem }() // release slot

        fmt.Printf("[%v] Task %d running (slot acquired)\n",
            time.Since(start).Round(time.Millisecond), id)
        time.Sleep(50 * time.Millisecond) // simulate work
    }(i)
}

wg.Wait()
fmt.Printf("\nAll 10 tasks done in %v (with max %d concurrent)\n",
    time.Since(start).Round(time.Millisecond), maxConcurrency)

## 10. Errgroup (Concurrent Tasks with Error Propagation)

In [ ]:
import (
    "context"
    "errors"
    "fmt"
    "sync"
    "time"
)

// Simplified errgroup (golang.org/x/sync/errgroup in production)
// Runs goroutines and returns the first error encountered.
type ErrGroup struct {
    wg   sync.WaitGroup
    mu   sync.Mutex
    err  error
    ctx  context.Context
    cancel context.CancelFunc
}

func NewErrGroup(ctx context.Context) *ErrGroup {
    ctx, cancel := context.WithCancel(ctx)
    return &ErrGroup{ctx: ctx, cancel: cancel}
}

func (g *ErrGroup) Go(f func(ctx context.Context) error) {
    g.wg.Add(1)
    go func() {
        defer g.wg.Done()
        if err := f(g.ctx); err != nil {
            g.mu.Lock()
            if g.err == nil {
                g.err = err
                g.cancel() // cancel other goroutines
            }
            g.mu.Unlock()
        }
    }()
}

func (g *ErrGroup) Wait() error {
    g.wg.Wait()
    g.cancel()
    return g.err
}

// Demo: fetch multiple "services" concurrently, one fails
eg := NewErrGroup(context.Background())

services := []string{"users", "orders", "payments", "inventory"}

for _, svc := range services {
    svc := svc
    eg.Go(func(ctx context.Context) error {
        time.Sleep(30 * time.Millisecond)

        // Simulate payments service failure
        if svc == "payments" {
            return errors.New("payments: connection refused")
        }

        select {
        case <-ctx.Done():
            fmt.Printf("  %s: cancelled\n", svc)
            return ctx.Err()
        default:
            fmt.Printf("  %s: OK\n", svc)
            return nil
        }
    })
}

if err := eg.Wait(); err != nil {
    fmt.Printf("\nGroup failed: %v\n", err)
} else {
    fmt.Println("\nAll services OK")
}

## 11. Or-Done Channel (First Result Wins)

In [ ]:
import (
    "fmt"
    "time"
)

// Or pattern: given N channels, return a channel that closes
// when ANY one of them closes. Useful for racing cancellation signals.
func or(channels ...<-chan struct{}) <-chan struct{} {
    switch len(channels) {
    case 0:
        return nil
    case 1:
        return channels[0]
    }

    done := make(chan struct{})
    go func() {
        defer close(done)
        switch len(channels) {
        case 2:
            select {
            case <-channels[0]:
            case <-channels[1]:
            }
        default:
            select {
            case <-channels[0]:
            case <-channels[1]:
            case <-channels[2]:
            case <-or(append(channels[3:], done)...):
            }
        }
    }()
    return done
}

// Helper: returns a channel that closes after duration
func after(d time.Duration) <-chan struct{} {
    ch := make(chan struct{})
    go func() {
        time.Sleep(d)
        close(ch)
    }()
    return ch
}

// Race three signals — whichever fires first wins
start := time.Now()
<-or(
    after(200 * time.Millisecond),
    after(500 * time.Millisecond),
    after(50 * time.Millisecond), // this one wins
)
fmt.Printf("First signal received after %v\n", time.Since(start).Round(time.Millisecond))

## 12. Pipeline with Cancellation

In [ ]:
import (
    "context"
    "fmt"
)

// Production pipeline: each stage respects context cancellation
// so the entire pipeline tears down cleanly.

func gen(ctx context.Context, nums ...int) <-chan int {
    out := make(chan int)
    go func() {
        defer close(out)
        for _, n := range nums {
            select {
            case out <- n:
            case <-ctx.Done():
                return
            }
        }
    }()
    return out
}

func double(ctx context.Context, in <-chan int) <-chan int {
    out := make(chan int)
    go func() {
        defer close(out)
        for n := range in {
            select {
            case out <- n * 2:
            case <-ctx.Done():
                return
            }
        }
    }()
    return out
}

func filter(ctx context.Context, in <-chan int, pred func(int) bool) <-chan int {
    out := make(chan int)
    go func() {
        defer close(out)
        for n := range in {
            if pred(n) {
                select {
                case out <- n:
                case <-ctx.Done():
                    return
                }
            }
        }
    }()
    return out
}

// Pipeline: generate → double → filter (keep > 10)
// Cancel early after collecting 3 results
ctx, cancel := context.WithCancel(context.Background())
defer cancel()

pipeline := filter(ctx,
    double(ctx,
        gen(ctx, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10)),
    func(n int) bool { return n > 10 },
)

collected := 0
for v := range pipeline {
    fmt.Printf("Pipeline output: %d\n", v)
    collected++
    if collected == 3 {
        cancel() // tear down the entire pipeline
        break
    }
}
fmt.Printf("Collected %d results, pipeline cancelled\n", collected)

## Try It Yourself

1. Build a concurrent web scraper that fetches 10 URLs in parallel with a semaphore limiting to 3 at a time.
2. Implement a request coalescer: if multiple goroutines ask for the same key simultaneously, only one fetch happens and all waiters get the result.
3. Create a pipeline with 4 stages (generate → validate → transform → collect) where any stage can be cancelled via context.
4. Build a circuit breaker: after 3 consecutive failures, stop calling the service for 5 seconds before retrying.

In [ ]:
// Your code here